## 1. Header1. Header & Description / 헤더 & 설명

This script trains a ResNet18 model for 4-class emotion recognition (angry, sad, neutral, happy).
- Uses multiple dataset paths merged with ConcatDataset.
- Includes upgraded preprocessing (Resize → Crop → Normalize).
- Supports advanced training features: AdamW optimizer, cosine scheduler, warmup, mixed precision (AMP), EMA (Exponential Moving Average), and Mixup augmentation.
- Evaluates after each epoch (Accuracy, Macro-F1, report, confusion matrix).
- Saves the best-performing model once at the end (no checkpoints during training).

이 스크립트는 ResNet18 기반 4클래스 감정 인식 모델(angry, sad, neutral, happy)을 학습합니다.
- 여러 데이터셋 경로를 ConcatDataset으로 통합.
- 데이터 전처리 업그레이드(Resize → Crop → Normalize).
- 학습 기능: AdamW 옵티마이저, CosineAnnealing 스케줄러, Warmup, 혼합 정밀도(AMP), EMA, Mixup.
- 매 epoch마다 정확도, Macro-F1, 분류 리포트, 혼동행렬 평가.
- 학습 중간 저장 없이, 최고 성능 모델만 마지막에 저장.

In [ ]:
import os, time, copy
from dataclasses import dataclass
from typing import Dict, List
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

## 2. Configuration / 설정

In [ ]:
@dataclass
class Cfg:
    target_classes: List[str] = ("angry", "sad", "neutral", "happy")
    img_size: int = 224
    batch_size: int = 64
    num_workers: int = 8
    epochs: int = 40
    warmup_epochs: int = 5
    lr: float = 3e-4
    weight_decay: float = 0.05
    dropout: float = 0.3
    seed: int = 42

    use_mixup: bool = True
    mixup_alpha: float = 0.2
    use_ema: bool = True
    ema_decay: float = 0.999
    early_patience: int = 7

    use_weighted_sampler: bool = True
    label_smoothing: float = 0.1

    # 경로(사용자 환경 유지)
    base_path_1: str = "/Users/star/Desktop/DSWU/2025/code/emotion_cnn/archive"
    base_path_2: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/archive"
    base_path_3: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/affectnet_filtered"
    save_path: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/new_Emotion_Resnet.pth"

cfg = Cfg()

Defines all hyperparameters and paths using a dataclass.
- Classes: angry, sad, neutral, happy.
- Training setup: 40 epochs, lr=3e-4, batch=64, warmup=5 epochs.
- Features: Mixup, EMA, Label smoothing, Weighted sampler.
- Dataset paths: three dataset bases are used.
- Saves final model to save_path.

dataclass를 사용해 하이퍼파라미터 및 경로를 정의합니다.
- 감정 클래스: angry, sad, neutral, happy.
- 학습 설정: 40 epoch, 학습률 3e-4, 배치 크기 64, warmup 5 epoch.
- 사용 기능: Mixup, EMA, Label smoothing, 가중치 샘플링.
- 데이터셋: 3개 base 폴더 경로 지정.
- 최종 모델은 save_path에 저장됩니다.

## 3. Seed & Device / 시드 & 장치 설정

In [ ]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
set_seed(cfg.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

- Sets random seeds for reproducibility and selects GPU (cuda) if available, otherwise CPU.
- 재현성을 위해 랜덤 시드를 고정하고, 가능하면 GPU(cuda), 아니면 CPU를 선택합니다

## 4. Data Transforms / 데이터 전처리

In [ ]:
train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(cfg.img_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02,0.15)),
])

test_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

Defines image preprocessing pipelines.
- Training: Resize → Crop → Flip → Color Jitter → Grayscale → Blur → Normalize → Random Erasing.
- Testing: Only Resize, Crop, Normalize.
이미지 전처리 파이프라인 정의.
- 학습용: Resize → Crop → Flip → 색상 변형 → 흑백 변환 → 블러 → 정규화 → 랜덤 지우기.
- 테스트용: Resize, Crop, Normalize만 수행.

## 5. Dataset Loading & Label Remapping / 데이터셋 로딩 & 라벨 리맵핑

In [ ]:
new_class_to_idx: Dict[str,int] = {c:i for i,c in enumerate(cfg.target_classes)}

def remap_labels(imagefolder_ds: datasets.ImageFolder,
                 original_class_to_idx: Dict[str, int],
                 new_class_to_idx: Dict[str, int]):
    new_samples, new_labels = [], []
    inv = {v:k for k,v in original_class_to_idx.items()}
    for path, label in imagefolder_ds.samples:
        cname = inv[label]
        if cname in new_class_to_idx:
            nl = new_class_to_idx[cname]
            new_samples.append((path, nl))
            new_labels.append(nl)
    imagefolder_ds.samples = new_samples
    imagefolder_ds.targets = new_labels
    imagefolder_ds.class_to_idx = new_class_to_idx
    return imagefolder_ds, new_labels


- ImageFolder datasets are remapped to ensure only the target 4 classes are used (angry, sad, neutral, happy).
- ImageFolder 데이터셋의 라벨을 리맵핑하여 4개 타겟 클래스(angry, sad, neutral, happy)만 사용하도록 정리합니다.

In [ ]:
bp1, bp2, bp3 = cfg.base_path_1, cfg.base_path_2, cfg.base_path_3

train_ds1 = datasets.ImageFolder(os.path.join(bp1, 'train'), transform=train_tf)
train_ds2 = datasets.ImageFolder(os.path.join(bp2, 'train'), transform=train_tf)
train_ds3 = datasets.ImageFolder(os.path.join(bp3, 'train'), transform=train_tf)
train_ds1, l1 = remap_labels(train_ds1, train_ds1.class_to_idx, new_class_to_idx)
train_ds2, l2 = remap_labels(train_ds2, train_ds2.class_to_idx, new_class_to_idx)
train_ds3, l3 = remap_labels(train_ds3, train_ds3.class_to_idx, new_class_to_idx)
full_train = ConcatDataset([train_ds1, train_ds2, train_ds3])
all_train_labels = l1 + l2 + l3

# test
test_ds1 = datasets.ImageFolder(os.path.join(bp1, 'test'), transform=test_tf)
test_ds2 = datasets.ImageFolder(os.path.join(bp2, 'test'), transform=test_tf)
test_ds3 = datasets.ImageFolder(os.path.join(bp3, 'test'), transform=test_tf)
test_ds1, _ = remap_labels(test_ds1, test_ds1.class_to_idx, new_class_to_idx)
test_ds2, _ = remap_labels(test_ds2, test_ds2.class_to_idx, new_class_to_idx)
test_ds3, _ = remap_labels(test_ds3, test_ds3.class_to_idx, new_class_to_idx)
full_test = ConcatDataset([test_ds1, test_ds2, test_ds3])

6. DataLoader with Weighted Sampler / 데이터로더 & 가중치 샘플러

In [ ]:
if cfg.use_weighted_sampler:
    cnt = Counter(all_train_labels)
    class_w = [1.0/max(cnt.get(i,1),1) for i in range(len(cfg.target_classes))]
    sample_w = []
    for ds in [train_ds1, train_ds2, train_ds3]:
        sample_w.extend([class_w[y] for _, y in ds.samples])
    sampler = WeightedRandomSampler(weights=sample_w, num_samples=len(sample_w), replacement=True)
    train_loader = DataLoader(full_train, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)
else:
    train_loader = DataLoader(full_train, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=True, drop_last=True)

test_loader = DataLoader(full_test, batch_size=cfg.batch_size, shuffle=False,
                         num_workers=cfg.num_workers, pin_memory=True)

- To handle class imbalance, uses WeightedRandomSampler so that each class is sampled more evenly.
- 클래스 불균형 문제를 완화하기 위해 WeightedRandomSampler를 사용하여 모든 클래스가 균등하게 학습되도록 합니다.

## 7. Model Definition / 모델 정의

In [ ]:
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
for name, p in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

in_f = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(in_f, 128), nn.ReLU(inplace=True), nn.Dropout(cfg.dropout), nn.Linear(128, len(cfg.target_classes))
)
model = model.to(device)

Loads a ResNet18 pretrained model.
- Freezes all layers except the last block (layer4) and the classifier head (fc).
- Replaces final fully-connected layer with: Linear → ReLU → Dropout → Linear(→4 classes).

  
ResNet18 사전학습 모델을 불러옵니다.
- layer4와 fc를 제외한 나머지 레이어는 동결.
- 마지막 fc를 교체: Linear → ReLU → Dropout → Linear(출력=4클래스).

## 8. Loss, Optimizer, Scheduler, EMA, Mixup / 손실함수, 옵티마이저, EMA, Mixup

In [ ]:
class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.1):
        super().__init__()
        self.eps = eps
    def forward(self, logits, target):
        n = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true = torch.zeros_like(logp).fill_(self.eps/(n-1))
            true.scatter_(1, target.unsqueeze(1), 1-self.eps)
        return torch.mean(torch.sum(-true*logp, dim=1))

criterion = LabelSmoothingCE(eps=cfg.label_smoothing)
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
scaler = GradScaler()

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.clone() for n,p in model.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data*(1-self.decay))
    @torch.no_grad()
    def apply_to(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])

ema = EMA(model, decay=cfg.ema_decay) if cfg.use_ema else None

# mixup
def mixup_data(x, y, alpha=0.2):
    if alpha is None or alpha <= 0:
        return x, y, 1.0, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, lam, y_b

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

- LabelSmoothingCE: makes training less sensitive to noisy labels.
- AdamW: optimizer with weight decay.
- CosineAnnealingLR: learning rate scheduler.
- GradScaler: enables AMP (mixed precision).  
<br/>

- LabelSmoothingCE: 잘못된 라벨에 덜 민감하게 학습.
- AdamW: weight decay 포함 옵티마이저.
- CosineAnnealingLR: 학습률 스케줄러.
- GradScaler: AMP(혼합 정밀도 학습) 사용.

EMA & Mixup Functions:
- EMA: keeps a moving average of weights for stable evaluation.
- Mixup: blends two samples and labels to improve generalization.

## 9. Evaluation Function / 평가 함수

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    macroF1 = f1_score(all_labels, all_preds, average='macro')
    print(f"\t✅ Val Acc: {acc*100:.2f}% | Macro-F1: {macroF1*100:.2f}%")
    print(classification_report(all_labels, all_preds, target_names=list(cfg.target_classes), digits=4))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))
    return acc, macroF1


- Evaluates model with Accuracy and Macro-F1, prints classification report and confusion matrix.
- 모델을 정확도, Macro-F1로 평가하고 분류 리포트 및 혼동행렬 출력.

## 10. Training Loop / 학습 루프

In [ ]:
best_f1 = -1.0
wait = 0
best_state = None  # 마지막에 1회 저장할 state

for epoch in range(1, cfg.epochs+1):
    # warmup
    if epoch <= cfg.warmup_epochs:
        warm_lr = cfg.lr * epoch / max(cfg.warmup_epochs,1)
        for g in optimizer.param_groups: g['lr'] = warm_lr

    model.train()
    running = 0.0
    start = time.time()

    pbar = tqdm(train_loader, desc=f"Epoch [{epoch}/{cfg.epochs}]", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        if cfg.use_mixup:
            images, y_a, lam, y_b = mixup_data(images, labels, alpha=cfg.mixup_alpha)

        optimizer.zero_grad(set_to_none=True)
        with autocast():
            outputs = model(images)
            if cfg.use_mixup:
                loss = mixup_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        if ema:
            ema.update(model)

    avg_loss = running / max(len(train_loader),1)
    print(f"[Epoch {epoch}] ✅ Avg Loss: {avg_loss:.4f} | Time: {time.time()-start:.2f}s")

    # EMA 가중치로 평가(있으면)
    if ema:
        backup = copy.deepcopy(model.state_dict())
        ema.apply_to(model)
        val_acc, val_f1 = evaluate(model, test_loader)
        # best state는 EMA 적용된 가중치로 보관
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        model.load_state_dict(backup)
    else:
        val_acc, val_f1 = evaluate(model, test_loader)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

    scheduler.step()

    if wait >= cfg.early_patience:
        print("\t>> Early stopped.")
        break

- Warmup learning rate.
- Training loop with AMP + GradScaler.
- Uses Mixup if enabled.
- EMA applied for evaluation.
- Early stopping if no improvement.  
<br/>  

- Warmup으로 학습률 점진적 증가.
- AMP + GradScaler로 효율적 학습.
- Mixup 옵션 활성화 시 데이터 증강 적용.
- EMA 가중치로 검증.
- 성능 개선 없으면 Early stopping.

## 11. Save Best Model / 최고 성능 모델 저장

- Saves the best model (highest Macro-F1) at the end.
- 학습이 끝나면 최고 성능 모델(최대 Macro-F1 기준)을 저장.

In [ ]:

if best_state is None:
    # fallback: 현재 모델 저장
    best_state = model.state_dict()

torch.save(best_state, cfg.save_path)
print(f"\n✅ 최종 최고 성능 모델 저장 완료: {cfg.save_path}")
print(f"   >> Best Macro-F1: {best_f1*100:.2f}%")


/var/folders/09/n0nfs13n71x3xc6n3ln9bz9r0000gn/T/ipykernel_39875/4285003786.py:203: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/Users/star/anaconda3/envs/emotion-env/lib/python3.11/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(
Epoch [1/40]:   0%|                                     | 0/654 [00:00<?, ?it/s]/var/folders/09/n0nfs13n71x3xc6n3ln9bz9r0000gn/T/ipykernel_39875/4285003786.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
                                                                                

[Epoch 1] ✅ Avg Loss: 1.0939 | Time: 1220.78s


	✅ Val Acc: 54.02% | Macro-F1: 51.61%
              precision    recall  f1-score   support

       angry     0.4240    0.4705    0.4460      1120
         sad     0.8352    0.3467    0.4900      2411
     neutral     0.4181    0.9088    0.5727      2940
       happy     0.8891    0.4042    0.5558      3968

    accuracy                         0.5402     10439
   macro avg     0.6416    0.5326    0.5161     10439
weighted avg     0.6941    0.5402    0.5336     10439

Confusion Matrix:
 [[ 527   37  496   60]
 [ 321  836 1152  102]
 [ 159   71 2672   38]
 [ 236   57 2071 1604]]


Epoch [2/40]:   0%|                                     | 0/654 [00:00<?, ?it/s]/var/folders/09/n0nfs13n71x3xc6n3ln9bz9r0000gn/T/ipykernel_39875/4285003786.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/Users/star/anaconda3/envs/emotion-env/lib/python3.11/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
                                                                                

[Epoch 2] ✅ Avg Loss: 0.9862 | Time: 1220.50s


...

[Epoch 22] ✅ Avg Loss: 0.7052 | Time: 1224.32s


	✅ Val Acc: 82.68% | Macro-F1: 79.84%
              precision    recall  f1-score   support

       angry     0.7001    0.7107    0.7054      1120
         sad     0.7532    0.8051    0.7783      2411
     neutral     0.8071    0.7769    0.7917      2940
       happy     0.9268    0.9098    0.9182      3968

    accuracy                         0.8268     10439
   macro avg     0.7968    0.8006    0.7984     10439
weighted avg     0.8287    0.8268    0.8274     10439

Confusion Matrix:
 [[ 796  185   89   50]
 [ 165 1941  235   70]
 [ 116  375 2284  165]
 [  60   76  222 3610]]


Epoch [23/40]:   0%|                                    | 0/654 [00:00<?, ?it/s]/var/folders/09/n0nfs13n71x3xc6n3ln9bz9r0000gn/T/ipykernel_39875/4285003786.py:279: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/Users/star/anaconda3/envs/emotion-env/lib/python3.11/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
                                                                                

[Epoch 23] ✅ Avg Loss: 0.6850 | Time: 1222.14s


	✅ Val Acc: 82.61% | Macro-F1: 79.74%
              precision    recall  f1-score   support

       angry     0.6437    0.7580    0.6962      1120
         sad     0.8077    0.7578    0.7819      2411
     neutral     0.7884    0.8024    0.7953      2940
       happy     0.9283    0.9045    0.9163      3968

    accuracy                         0.8261     10439
   macro avg     0.7920    0.8057    0.7974     10439
weighted avg     0.8305    0.8261    0.8276     10439

Confusion Matrix:
 [[ 849  120  111   40]
 [ 228 1827  277   79]
 [ 153  270 2359  158]
 [  89   45  245 3589]]
	>> Early stopped.

✅ 최종 최고 성능 모델 저장 완료: /Users/star/Desktop/DSWU/2025/code/emotion_resnet/new_Emotion_Resnet.pth
   >> Best Macro-F1: 80.24%


## 1. Header & Overview / 헤더 & 개요

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
한 번에 실행해서 s2 → s3a → s3b를 학습/평가하고, 
metrics를 비교해 **최고 성능 교사(teacher) .pth**를 자동 선택하는 스크립트.

옵션으로 s4(풀파인튜닝)·s5(도메인 튜닝)도 같은 런에서 이어서 수행 가능.
- 외부 셸 커맨드 없이 파이썬만으로 디렉토리 생성/저장/비교까지 처리
- 내부에 간소화된 학습 루프(AdamW + Cosine + Warmup + EMA + Mixup + WeightedSampler)
- 증강 강도, 배치, 에폭 등 상단 파라미터로 손쉽게 조정

실행:
  python auto_teacher_selection.py
끝나면 화면에 ✅ 최종 교사 모델 경로와 F1이 출력됩니다.
"""

This script trains three different models (s2, s3a, s3b) in sequence and automatically selects the best-performing one as the teacher model.
- s2: ResNet18 (layer3+4+head).
- s3a: ResNet34 (layer3+4+head).
- s3b: EfficientNet-B0 (last stage + classifier).
<br/>

  Optional:
- s4: Full fine-tuning.
- s5: Domain adaptation training.
<br/>

이 스크립트는 s2, s3a, s3b 세 가지 모델을 순차적으로 학습/평가하고, 가장 성능이 좋은 모델을 교사(teacher) 모델로 자동 선택합니다.
- s2: ResNet18 (layer3+4+head).
- s3a: ResNet34 (layer3+4+head).
- s3b: EfficientNet-B0 (마지막 stage + classifier).
<br/>

  옵션:
- s4: 전체 파라미터 풀파인튜닝.
- s5: 도메인 데이터 튜닝.

In [ ]:
import os, json, time, copy
from dataclasses import dataclass
from typing import Dict, List, Optional
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from contextlib import nullcontext

from torchvision import datasets, transforms as T, models
from torchvision.models import (
    ResNet18_Weights, ResNet34_Weights, EfficientNet_B0_Weights
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

## 2. User Configuration / 사용자 설정

In [ ]:
EXP_DIR = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810"  # 결과 저장 폴더
DOMAIN_PATH: Optional[str] = None  # 도메인 튜닝 데이터(ImageFolder, train/test)
RUN_S4 = False   # s2/s3a/s3b 후, 선택된 백본에 대해 풀파인튜닝까지 수행할지
RUN_S5 = False   # s5 도메인 튜닝 수행할지 (DOMAIN_PATH 필요)
SEED = 42

- EXP_DIR: output directory for experiment results.
- DOMAIN_PATH: optional dataset path for domain tuning.
- RUN_S4, RUN_S5: toggle whether to run fine-tuning or domain adaptation.  
<br/>

- EXP_DIR: 실험 결과 저장 경로.
- DOMAIN_PATH: 도메인 튜닝용 데이터셋 경로.
- RUN_S4, RUN_S5: 풀파인튜닝/도메인 튜닝 실행 여부 설정.

## 3. Hyperparameters / 공통 하이퍼파라미터

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 8
EPOCHS_S2 = 35
EPOCHS_S3 = 35
EPOCHS_S4 = 12
EPOCHS_S5 = 8
WARMUP_S2 = 5
WARMUP_S3 = 5
WARMUP_S4 = 2
WARMUP_S5 = 2
BASE_LR_S2 = 3e-4
BASE_LR_S3 = 2.5e-4
BASE_LR_S4 = 1e-4
BASE_LR_S5 = 1.5e-4
DROPOUT_RES18 = 0.3
DROPOUT_RES34 = 0.4
DROPOUT_EFFB0 = 0.4
LABEL_SMOOTH = 0.1
MIXUP_ALPHA = 0.2
USE_EMA = True
EMA_DECAY = 0.999
USE_SAMPLER = True

- Image size: 224, batch size: 64.
- Epochs: s2=35, s3=35, s4=12, s5=8.
- Techniques: label smoothing, mixup, EMA, weighted sampler.  
<br/>

- 이미지 크기: 224, 배치 크기: 64.
- 학습 epoch: s2=35, s3=35, s4=12, s5=8.
- 학습 기법: 라벨 스무딩, Mixup, EMA, 가중치 샘플링.

In [ ]:
# 데이터 경로(사용자 환경)
BASE1 = "/Users/star/Desktop/DSWU/2025/code/emotion_cnn/archive"
BASE2 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/archive"
BASE3 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/affectnet_filtered"
TARGET_CLASSES = ("angry", "sad", "neutral", "happy")

## 4. Utilities / 유틸 함수

In [ ]:
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = torch.cuda.is_available()

def select_device():
    if torch.cuda.is_available(): return 'cuda'
    elif torch.backends.mps.is_available(): return 'mps'
    else: return 'cpu'

new_class_to_idx: Dict[str,int] = {c:i for i,c in enumerate(TARGET_CLASSES)}

def make_transforms(img_size=224, aug_extra=True):
    if aug_extra:
        train_tf = T.Compose([
            T.Resize(256), T.RandomResizedCrop(img_size, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(0.5), T.ColorJitter(0.3,0.3,0.3,0.1),
            T.RandomGrayscale(0.1), T.GaussianBlur(3, sigma=(0.1,2.0)),
            T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            T.RandomErasing(p=0.25, scale=(0.02,0.15)),
        ])
    else:
        train_tf = T.Compose([
            T.Resize(256), T.CenterCrop(img_size), T.RandomHorizontalFlip(0.5),
            T.ColorJitter(0.3,0.3,0.3,0.1), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
    test_tf = T.Compose([
        T.Resize(256), T.CenterCrop(img_size), T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    return train_tf, test_tf


def remap_labels(ds: datasets.ImageFolder):
    inv = {v:k for k,v in ds.class_to_idx.items()}
    samples, labels = [], []
    for path, y in ds.samples:
        cname = inv[y]
        if cname in new_class_to_idx:
            ny = new_class_to_idx[cname]
            samples.append((path, ny)); labels.append(ny)
    ds.samples = samples; ds.targets = labels; ds.class_to_idx = new_class_to_idx
    return ds, labels


def load_concat(train_tf, test_tf, include_domain=False):
    train_sets, test_sets, all_y = [], [], []
    for bp in [BASE1, BASE2, BASE3]:
        tr = datasets.ImageFolder(os.path.join(bp,'train'), transform=train_tf)
        tr, ls = remap_labels(tr); train_sets.append(tr); all_y+=ls
        te = datasets.ImageFolder(os.path.join(bp,'test'), transform=test_tf)
        te, _  = remap_labels(te); test_sets.append(te)
    if include_domain and DOMAIN_PATH:
        dp = DOMAIN_PATH
        tr = datasets.ImageFolder(os.path.join(dp,'train'), transform=train_tf)
        tr, ls = remap_labels(tr); train_sets.append(tr); all_y+=ls
        te = datasets.ImageFolder(os.path.join(dp,'test'), transform=test_tf)
        te, _  = remap_labels(te); test_sets.append(te)
    return ConcatDataset(train_sets), ConcatDataset(test_sets), all_y


def make_loaders(full_train, full_test, batch=BATCH_SIZE, device_type='cpu'):
    if USE_SAMPLER:
        all_labels = []
        for ds in full_train.datasets:
            all_labels.extend(ds.targets)
        cnt = Counter(all_labels)
        class_w = [1.0/max(cnt.get(i,1),1) for i in range(len(TARGET_CLASSES))]
        sample_w = []
        for ds in full_train.datasets:
            sample_w += [class_w[y] for _, y in ds.samples]
        sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)
        train_loader = DataLoader(full_train, batch_size=batch, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=(device_type=='cuda'), drop_last=True)
    else:
        train_loader = DataLoader(full_train, batch_size=batch, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=(device_type=='cuda'), drop_last=True)
    val_loader = DataLoader(full_test, batch_size=batch, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(device_type=='cuda'))
    return train_loader, val_loader

- set_seed(): reproducibility.
- select_device(): selects cuda, mps, or cpu.
- make_transforms(): defines training/test augmentations.
- remap_labels(): remaps dataset labels to target classes.
- load_concat(): loads and concatenates multiple datasets.
- make_loaders(): creates DataLoaders with optional weighted sampling.  
<br/>

- set_seed(): 재현성 확보.
- select_device(): cuda, mps, cpu 중 자동 선택.
- make_transforms(): 학습/테스트용 이미지 전처리 정의.
- remap_labels(): 데이터셋 라벨을 타겟 클래스에 맞게 리맵핑.
- load_concat(): 여러 데이터셋을 불러와 ConcatDataset으로 병합.
- make_loaders(): DataLoader 생성 (가중치 샘플링 가능).

## 5. Model & Training Setup / 모델 & 학습 구성

In [ ]:
def build_model(backbone: str, num_classes: int, dropout=0.3):
    if backbone == 'resnet18':
        m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        in_f = m.fc.in_features
        m.fc = nn.Sequential(nn.Linear(in_f, 128), nn.ReLU(True), nn.Dropout(dropout), nn.Linear(128, num_classes))
        return m
    if backbone == 'resnet34':
        m = models.resnet34(weights=ResNet34_Weights.DEFAULT)
        in_f = m.fc.in_features
        m.fc = nn.Sequential(nn.Linear(in_f, 256), nn.ReLU(True), nn.Dropout(max(0.4, dropout)), nn.Linear(256, num_classes))
        return m
    if backbone == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        in_f = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_f, num_classes)
        if isinstance(m.classifier[0], nn.Dropout): m.classifier[0].p = max(0.3, dropout)
        return m
    raise ValueError('unknown backbone')


def set_trainable(model: nn.Module, backbone: str, scope: str):
    for p in model.parameters(): p.requires_grad = False
    if backbone.startswith('resnet'):
        if scope == 'l4_fc':
            for n,p in model.named_parameters():
                if 'layer4' in n or 'fc' in n: p.requires_grad = True
        elif scope == 'l3_l4_fc':
            for n,p in model.named_parameters():
                if 'layer3' in n or 'layer4' in n or 'fc' in n: p.requires_grad = True
        elif scope == 'full':
            for p in model.parameters(): p.requires_grad = True
    elif backbone == 'efficientnet_b0':
        if scope == 'l4_fc':
            for n,p in model.named_parameters():
                if n.startswith('classifier.'): p.requires_grad = True
        elif scope == 'l3_l4_fc':
            for n,p in model.named_parameters():
                if n.startswith('features.6') or n.startswith('features.7') or n.startswith('classifier.'):
                    p.requires_grad = True
        elif scope == 'full':
            for p in model.parameters(): p.requires_grad = True

class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.1): super().__init__(); self.eps=eps
    def forward(self, logits, target):
        n = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true = torch.zeros_like(logp).fill_(self.eps/(n-1))
            true.scatter_(1, target.unsqueeze(1), 1-self.eps)
        return torch.mean(torch.sum(-true*logp, dim=1))

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n:p.data.clone() for n,p in model.named_parameters() if p.requires_grad}
    @torch.no_grad()
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data*(1-self.decay))
    @torch.no_grad()
    def apply_to(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.shadow[n])


def evaluate(model, loader, device, classes):
    model.eval(); all_p, all_y = [], []
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)
            pred = logits.argmax(1)
            all_p.extend(pred.cpu().numpy()); all_y.extend(y.cpu().numpy())
    acc = accuracy_score(all_y, all_p)
    f1  = f1_score(all_y, all_p, average='macro')
    print(f"\tVal Acc: {acc*100:.2f}% | Macro-F1: {f1*100:.2f}%")
    print(classification_report(all_y, all_p, target_names=list(classes), digits=4))
    print("Confusion Matrix:\n", confusion_matrix(all_y, all_p))
    return acc, f1

- build_model(): builds ResNet18, ResNet34, or EfficientNet-B0 with custom classifier.
- set_trainable(): controls which layers are trainable (l4_fc, l3_l4_fc, full).
- LabelSmoothingCE: cross-entropy with label smoothing.
- EMA: maintains exponential moving average of weights.
- evaluate(): evaluates model (accuracy, F1, classification report, confusion matrix).  
<br/>

- build_model(): ResNet18/34, EfficientNet-B0을 생성하고 분류기 교체.
- set_trainable(): 학습할 레이어 범위 지정(l4_fc, l3_l4_fc, full).
- LabelSmoothingCE: 라벨 스무딩을 적용한 교차 엔트로피.
- EMA: 가중치 EMA 추적.
- evaluate(): 모델 평가(정확도, F1, 분류 리포트, 혼동행렬).

## 6. Training Function / 학습 함수

In [ ]:
def train_once(tag: str, backbone: str, scope: str, epochs: int, warmup: int, base_lr: float, dropout: float,
               include_domain=False):
    os.makedirs(EXP_DIR, exist_ok=True)
    set_seed(SEED)
    device_type = select_device(); device = torch.device(device_type)

    train_tf, test_tf = make_transforms(IMG_SIZE, aug_extra=True)
    full_train, full_val, _ = load_concat(train_tf, test_tf, include_domain=include_domain)
    train_loader, val_loader = make_loaders(full_train, full_val, BATCH_SIZE, device_type)

    model = build_model(backbone, len(TARGET_CLASSES), dropout).to(device)
    set_trainable(model, backbone, scope)

    # AMP(CUDA 전용)
    use_amp = (device_type=='cuda')
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    amp_ctx = torch.amp.autocast(device_type='cuda', dtype=torch.float16) if use_amp else nullcontext()

    criterion = LabelSmoothingCE(LABEL_SMOOTH)
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=base_lr, weight_decay=0.05)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    ema = EMA(model, EMA_DECAY) if USE_EMA else None

    best_f1, wait, best_state = -1.0, 0, None

    def mixup_data(x, y, alpha=MIXUP_ALPHA):
        if alpha is None or alpha<=0: return x, y, 1.0, y
        lam = np.random.beta(alpha, alpha)
        idx = torch.randperm(x.size(0), device=x.device)
        return lam*x + (1-lam)*x[idx], y, lam, y[idx]
    def mixup_crit(pred, ya, yb, lam):
        return lam*criterion(pred, ya) + (1-lam)*criterion(pred, yb)

    for epoch in range(1, epochs+1):
        # warmup
        if epoch <= warmup:
            warm_lr = base_lr * epoch / max(warmup,1)
            for g in optimizer.param_groups: g['lr'] = warm_lr

        model.train(); running=0.0
        pbar = tqdm(train_loader, desc=f"{tag} Epoch [{epoch}/{epochs}]", leave=False)
        for x,y in pbar:
            x,y = x.to(device), y.to(device)
            x, ya, lam, yb = mixup_data(x,y)
            optimizer.zero_grad(set_to_none=True)
            with amp_ctx:
                logits = model(x)
                loss = mixup_crit(logits, ya, yb, lam)
            if use_amp:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            running += loss.item(); pbar.set_postfix(loss=f"{loss.item():.4f}")
            if ema: ema.update(model)
        print(f"[{tag}] Avg Loss: {running/max(1,len(train_loader)):.4f}")

        # eval(EMA 적용 상태로 평가)
        if ema:
            bk = copy.deepcopy(model.state_dict()); ema.apply_to(model)
            _, f1 = evaluate(model, val_loader, device, TARGET_CLASSES)
            if f1>best_f1: best_f1, wait, best_state = f1,0,copy.deepcopy(model.state_dict())
            else: wait+=1
            model.load_state_dict(bk)
        else:
            _, f1 = evaluate(model, val_loader, device, TARGET_CLASSES)
            if f1>best_f1: best_f1, wait, best_state = f1,0,copy.deepcopy(model.state_dict())
            else: wait+=1

        scheduler.step()
        if wait >= 7:
            print("\t>> Early stopped.")
            break

    if best_state is None: best_state = model.state_dict()
    save_pth = os.path.join(EXP_DIR, f"{tag}.pth")
    torch.save(best_state, save_pth)
    metrics = {
        'tag': tag, 'backbone': backbone, 'scope': scope,
        'epochs': epochs, 'warmup': warmup, 'lr': base_lr,
        'batch_size': BATCH_SIZE, 'best_macro_f1': float(best_f1)
    }
    with open(os.path.join(EXP_DIR, f"{tag}.metrics.json"), 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"\n✅ 저장: {save_pth}")
    print("📊 metrics:", metrics)
    return save_pth, best_f1


Trains one model (ResNet18, ResNet34, or EfficientNet-B0) for given epochs.
- Warmup LR.
- Mixup augmentation.
- AMP training for efficiency.
- EMA evaluation.
- Saves best model and metrics JSON.  
<br/>

하나의 모델(ResNet18, ResNet34, EfficientNet-B0)을 지정된 epoch 동안 학습.
- Warmup 학습률.
- Mixup 데이터 증강.
- AMP(혼합정밀도) 학습.
- EMA 가중치 평가.
- 최고 성능 모델 저장 + metrics JSON 기록.

## 7. Main Flow / 메인 실행 흐름

In [ ]:

if __name__ == '__main__':
    os.makedirs(EXP_DIR, exist_ok=True)
    set_seed(SEED)

    # s2: ResNet18, layer3+4+head
    p2, f2 = train_once(
        tag='s2', backbone='resnet18', scope='l3_l4_fc',
        epochs=EPOCHS_S2, warmup=WARMUP_S2, base_lr=BASE_LR_S2, dropout=DROPOUT_RES18
    )

    # s3a: ResNet34, layer3+4+head
    p3a, f3a = train_once(
        tag='s3a', backbone='resnet34', scope='l3_l4_fc',
        epochs=EPOCHS_S3, warmup=WARMUP_S3, base_lr=BASE_LR_S3, dropout=DROPOUT_RES34
    )

    # s3b: EfficientNet-B0, 마지막 스테이지+classifier
    p3b, f3b = train_once(
        tag='s3b', backbone='efficientnet_b0', scope='l3_l4_fc',
        epochs=EPOCHS_S3, warmup=WARMUP_S3, base_lr=BASE_LR_S3, dropout=DROPOUT_EFFB0
    )

    # 최고 성능 교사 선택
    scores = [('s2', p2, f2), ('s3a', p3a, f3a), ('s3b', p3b, f3b)]
    scores.sort(key=lambda x: x[2], reverse=True)
    best_tag, best_path, best_f1 = scores[0]
    print("\n==============================")
    print(f"✅ Teacher 후보 선택: {best_tag}.pth | Macro-F1={best_f1*100:.2f}%")
    print(f"경로: {best_path}")

    # 선택지: s4(풀파인튜닝)
    if RUN_S4:
        # 선택된 백본에 맞게 풀파인튜닝
        bb = {'s2':'resnet18', 's3a':'resnet34', 's3b':'efficientnet_b0'}[best_tag]
        p4, f4 = train_once(
            tag=f'{best_tag}_s4_full', backbone=bb, scope='full',
            epochs=EPOCHS_S4, warmup=WARMUP_S4, base_lr=BASE_LR_S4,
            dropout=(DROPOUT_RES18 if bb=='resnet18' else DROPOUT_RES34 if bb=='resnet34' else DROPOUT_EFFB0)
        )
        if f4 > best_f1:
            best_tag, best_path, best_f1 = f'{best_tag}_s4_full', p4, f4
            print(f"\n✨ s4로 갱신: {best_tag} | Macro-F1={best_f1*100:.2f}% | {best_path}")

    # 선택지: s5(도메인 튜닝)
    if RUN_S5 and DOMAIN_PATH:
        bb = {'s2':'resnet18', 's3a':'resnet34', 's3b':'efficientnet_b0'}[best_tag.split('_')[0]]
        p5, f5 = train_once(
            tag=f'{best_tag}_s5_domain', backbone=bb, scope='l3_l4_fc',
            epochs=EPOCHS_S5, warmup=WARMUP_S5, base_lr=BASE_LR_S5,
            dropout=(DROPOUT_RES18 if bb=='resnet18' else DROPOUT_RES34 if bb=='resnet34' else DROPOUT_EFFB0),
            include_domain=True
        )
        if f5 > best_f1:
            best_tag, best_path, best_f1 = f'{best_tag}_s5_domain', p5, f5
            print(f"\n✨ s5로 갱신: {best_tag} | Macro-F1={best_f1*100:.2f}% | {best_path}")

    print("\n==============================")
    print(f"🏁 최종 교사(teacher) 확정: {best_tag} | Macro-F1={best_f1*100:.2f}%")
    print(f"경로: {best_path}")
    print("이제 이 경로를 distill_emotion_teacher_student.py의 --teacher 인자로 사용하면 됩니다.")


[s2] Avg Loss: 1.0684


	Val Acc: 38.35% | Macro-F1: 34.20%
              precision    recall  f1-score   support

       angry     0.4749    0.4473    0.4607      1120
         sad     0.8488    0.1978    0.3209      2411
     neutral     0.3276    0.9592    0.4884      2940
       happy     0.9624    0.0517    0.0981      3968

    accuracy                         0.3835     10439
   macro avg     0.6534    0.4140    0.3420     10439
weighted avg     0.7051    0.3835    0.2984     10439

Confusion Matrix:
 [[ 501   10  608    1]
 [ 217  477 1711    6]
 [  90   29 2820    1]
 [ 247   46 3470  205]]


[s2] Avg Loss: 0.9613


	Val Acc: 65.70% | Macro-F1: 61.27%
              precision    recall  f1-score   support

       angry     0.3764    0.7857    0.5090      1120
         sad     0.9464    0.3078    0.4645      2411
     neutral     0.5691    0.8864    0.6932      2940
       happy     0.9606    0.6628    0.7844      3968

    accuracy                         0.6570     10439
   macro avg     0.7131    0.6607    0.6127     10439
weighted avg     0.7844    0.6570    0.6553     10439

Confusion Matrix:
 [[ 880    3  221   16]
 [ 681  742  922   66]
 [ 280   28 2606   26]
 [ 497   11  830 2630]]


[s2] Avg Loss: 0.9266


...

[s3b] Avg Loss: 0.7334


	Val Acc: 83.16% | Macro-F1: 80.39%
              precision    recall  f1-score   support

       angry     0.7031    0.7295    0.7160      1120
         sad     0.7704    0.7864    0.7783      2411
     neutral     0.8166    0.7905    0.8033      2940
       happy     0.9179    0.9183    0.9181      3968

    accuracy                         0.8316     10439
   macro avg     0.8020    0.8062    0.8039     10439
weighted avg     0.8323    0.8316    0.8318     10439

Confusion Matrix:
 [[ 817  163   91   49]
 [ 189 1896  254   72]
 [  92  319 2324  205]
 [  64   83  177 3644]]


[s3b] Avg Loss: 0.7222


	Val Acc: 82.95% | Macro-F1: 80.19%
              precision    recall  f1-score   support

       angry     0.7053    0.7223    0.7137      1120
         sad     0.7602    0.7943    0.7769      2411
     neutral     0.8123    0.7874    0.7997      2940
       happy     0.9228    0.9123    0.9175      3968

    accuracy                         0.8295     10439
   macro avg     0.8001    0.8041    0.8019     10439
weighted avg     0.8308    0.8295    0.8300     10439

Confusion Matrix:
 [[ 809  168   96   47]
 [ 186 1915  246   64]
 [  87  346 2315  192]
 [  65   90  193 3620]]


InterruptedError: [Errno 4] Interrupted system call

# Optimized train_once Script

## 1. Imports & Memory Guard / 임포트 및 메모리 가드

In [ ]:
import torch, gc, os
torch.set_num_threads(4)                 # CPU 스레드 과점 방지
torch.backends.cudnn.benchmark = False   # CPU에선 의미 적음, 안전하게 off

- These settings restrict PyTorch’s CPU thread usage and disable CUDNN benchmarking for stable runtime on low-resource or CPU-only systems.

- PyTorch의 CPU 스레드 사용량을 제한하고, CUDNN의 런타임 최적화 기능을 끕니다. CPU 환경에서 안정성을 확보하기 위한 설정입니다.

In [ ]:

def memory_guard():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

- Forces garbage collection and clears the CUDA cache to prevent memory fragmentation during long training loops.
- 학습 중 누적된 GPU 메모리 파편화를 방지하기 위해, 가비지 컬렉션 및 CUDA 캐시를 주기적으로 비웁니다.

## 2. make_loader Function / DataLoader 생성 함수

In [ ]:
def make_loader(dataset, batch_size, num_workers=0, shuffle=True):
    try:
        return torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, shuffle=shuffle,
            num_workers=num_workers, pin_memory=torch.cuda.is_available(),
            persistent_workers=False, drop_last=True if shuffle else False
        )
    except Exception as e:
        print(f"[Loader Fallback] {e} -> num_workers=0로 재시도")
        return torch.utils.data.DataLoader(
            dataset, batch_size=batch_size, shuffle=shuffle,
            num_workers=0, pin_memory=False, drop_last=True if shuffle else False
        )

- Creates a DataLoader with safe fallback — if multiprocessing workers cause errors, it retries with num_workers=0.

- DataLoader를 안전하게 생성합니다. 멀티프로세싱 에러 발생 시 자동으로 num_workers=0으로 재시도하도록 합니다.

## 3. train_once Function / 단일 학습 실행 함수

### Arguments & Overrides / 매개변수 및 오버라이드 옵션

In [ ]:
def train_once(
    tag: str,
    backbone: str,
    scope: str = 'l3_l4_fc',
    epochs: int = 35,
    warmup: int = 5,
    base_lr: float = 3e-4,
    dropout: float = 0.3,
    include_domain: bool = False,
    # ↓ 안전 모드 오버라이드
    override_img_size: int | None = None,
    override_batch_size: int | None = None,
    override_num_workers: int | None = None,
    mixup_alpha_override: float | None = None,
    # 임시 세이브(옵션)
    tmp_save_every: int | None = None
):
    import copy, time, numpy as np
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision import models, transforms, datasets
    from torchvision.models import ResNet18_Weights, ResNet34_Weights
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from torch.optim import AdamW
    from torch.optim.lr_scheduler import CosineAnnealingLR
    from torch.cuda.amp import autocast, GradScaler
    from tqdm import tqdm

- train_once() trains one model with optional overrides for batch size, image size, workers, and mixup alpha.
- It also supports temporary checkpoint saving every few epochs.  
<br/>

- train_once()는 단일 모델을 학습하며, 이미지 크기, 배치 크기, 워커 수, Mixup 파라미터를 오버라이드할 수 있습니다.
- 또한 일정 주기마다 임시 체크포인트를 저장하는 기능을 포함합니다.

### Transform Section / 데이터 전처리

In [ ]:
 # 하이퍼/전처리 확정
    IMG = override_img_size if override_img_size is not None else 224
    BATCH = override_batch_size if override_batch_size is not None else 64
    NW    = override_num_workers if override_num_workers is not None else 8
    MIXA  = mixup_alpha_override if mixup_alpha_override is not None else 0.2

    # ✅ CPU 친화 경량 증강(빠른 모드)
    train_tf = transforms.Compose([
        transforms.Resize(224 if IMG<=192 else 256),
        transforms.CenterCrop(IMG),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
        transforms.RandomGrayscale(0.05),
        # GaussianBlur 제거(무거움)
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02,0.12)),
    ])
    test_tf = transforms.Compose([
        transforms.Resize(224 if IMG<=192 else 256),
        transforms.CenterCrop(IMG),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

- Uses lightweight augmentations suitable for CPU — excludes Gaussian blur for speed, but retains color jitter, random grayscale, and random erasing. 
- CPU 환경에서 빠르게 동작하도록 경량화된 증강을 사용합니다.
GaussianBlur는 제거했으며, 색상 변형(ColorJitter), 랜덤 흑백, Erasing 등은 유지합니다.

### Dataset Loading & Label Remapping / 데이터셋 로드 및 라벨 리맵핑

In [ ]:
    # 데이터 로딩 + 라벨 리맵
    target = ['angry','sad','neutral','happy']
    new_c2i = {c:i for i,c in enumerate(target)}
    def remap(ds):
        inv = {v:k for k,v in ds.class_to_idx.items()}
        ns, nl = [], []
        for p, y in ds.samples:
            cn = inv[y]
            if cn in new_c2i:
                ny = new_c2i[cn]; ns.append((p, ny)); nl.append(ny)
        ds.samples, ds.targets, ds.class_to_idx = ns, nl, new_c2i
        return ds, nl

    bp1 = "/Users/star/Desktop/DSWU/2025/code/emotion_cnn/archive"
    bp2 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/archive"
    bp3 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/affectnet_filtered"

    tr1 = datasets.ImageFolder(os.path.join(bp1,'train'), transform=train_tf); tr1,_ = remap(tr1)
    tr2 = datasets.ImageFolder(os.path.join(bp2,'train'), transform=train_tf); tr2,_ = remap(tr2)
    tr3 = datasets.ImageFolder(os.path.join(bp3,'train'), transform=train_tf); tr3,_ = remap(tr3)
    te1 = datasets.ImageFolder(os.path.join(bp1,'test'),  transform=test_tf ); te1,_ = remap(te1)
    te2 = datasets.ImageFolder(os.path.join(bp2,'test'),  transform=test_tf ); te2,_ = remap(te2)
    te3 = datasets.ImageFolder(os.path.join(bp3,'test'),  transform=test_tf ); te3,_ = remap(te3)

    from torch.utils.data import ConcatDataset
    full_train = ConcatDataset([tr1,tr2,tr3])
    full_test  = ConcatDataset([te1,te2,te3])

    train_loader = make_loader(full_train, BATCH, NW, shuffle=True)
    test_loader  = make_loader(full_test , BATCH, NW, shuffle=False)


- Loads 3 dataset roots (emotion_cnn, emotion_resnet, affectnet_filtered), remaps all labels to four consistent emotion classes, and concatenates them using ConcatDataset.

- 3개의 데이터 경로(emotion_cnn, emotion_resnet, affectnet_filtered)에서 데이터를 불러오고,
모든 라벨을 4개 감정 클래스(angry, sad, neutral, happy)로 통일한 뒤 ConcatDataset으로 병합합니다.

### Model Building / 모델 구성

In [ ]:
    # 모델
    if backbone=='resnet18':
        model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    elif backbone=='resnet34':
        model = models.resnet34(weights=ResNet34_Weights.DEFAULT)
    elif backbone=='efficientnet_b0':
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    else:
        raise ValueError(backbone)

- Supports ResNet18, ResNet34, and EfficientNet-B0.
- Depending on the backbone, the classifier head is replaced to output 4 emotion classes.  
<br/>

- ResNet18, ResNet34, EfficientNet-B0을 지원하며,
- 백본별로 마지막 분류기를 교체하여 4개 감정 클래스를 출력하도록 수정합니다.

### Training Setup / 학습 구성 요소

In [ ]:
    # 학습 범위
    for n,p in model.named_parameters(): p.requires_grad = True
    if scope=='l3_l4_fc':
        for n,p in model.named_parameters():
            ok = any(k in n for k in ['layer3','layer4','classifier','fc'])
            p.requires_grad = ok

    # 헤드 교체
    if hasattr(model,'fc') and isinstance(model.fc, nn.Linear):
        in_f = model.fc.in_features
        model.fc = nn.Sequential(nn.Linear(in_f,128), nn.ReLU(True), nn.Dropout(dropout), nn.Linear(128,4))
    elif hasattr(model,'classifier') and isinstance(model.classifier, nn.Sequential):
        in_f = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_f, 4)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    # 손실/최적화/스케줄러/AMP/EMA
    class LSCE(nn.Module):
        def __init__(self, eps=0.1): super().__init__(); self.eps=eps
        def forward(self, logits, target):
            n = logits.size(1)
            logp = F.log_softmax(logits, dim=1)
            with torch.no_grad():
                true = torch.zeros_like(logp).fill_(self.eps/(n-1))
                true.scatter_(1, target.unsqueeze(1), 1-self.eps)
            return torch.mean(torch.sum(-true*logp, dim=1))
    criterion = LSCE(0.1)

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=base_lr, weight_decay=0.05)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    def mixup_data(x, y, a=MIXA):
        if a is None or a<=0: return x,y,1.0,y
        lam = np.random.beta(a,a); idx = torch.randperm(x.size(0), device=x.device)
        return lam*x + (1-lam)*x[idx], y, lam, y[idx]
    def mixup_crit(pred, ya, yb, lam): return lam*criterion(pred, ya) + (1-lam)*criterion(pred, yb)


- Loss: Label Smoothing Cross Entropy
- Optimizer: AdamW
- Scheduler: CosineAnnealingLR
- Techniques: Mixed precision (AMP), Mixup, EMA, Warmup.  
<br/>

- 손실 함수: 라벨 스무딩 교차 엔트로피
- 최적화: AdamW
- 스케줄러: CosineAnnealingLR
- 적용 기법: AMP(혼합 정밀도), Mixup, EMA, Warmup

### EMA Class / 지수이동평균(EMA)

In [ ]:
    class EMA:
        def __init__(self, m, decay=0.999): self.sh={n:p.data.clone() for n,p in m.named_parameters() if p.requires_grad}; self.decay=decay
        @torch.no_grad()
        def update(self, m):
            for n,p in m.named_parameters():
                if p.requires_grad: self.sh[n].mul_(self.decay).add_(p.data*(1-self.decay))
        @torch.no_grad()
        def apply_to(self, m):
            for n,p in m.named_parameters():
                if p.requires_grad: p.data.copy_(self.sh[n])
    ema = EMA(model, 0.999)

    @torch.no_grad()
    def evaluate(m):
        m.eval(); ap, ay = [], []
        for x,y in test_loader:
            x,y = x.to(device), y.to(device)
            ap += m(x).argmax(1).cpu().tolist()
            ay += y.cpu().tolist()
        acc = accuracy_score(ay, ap); f1 = f1_score(ay, ap, average='macro')
        print(f"\tVal Acc: {acc*100:.2f}% | Macro-F1: {f1*100:.2f}%")
        print(classification_report(ay, ap, target_names=target, digits=4))
        print("Confusion Matrix:\n", confusion_matrix(ay, ap))
        return acc, f1

    best_f1, best_state, wait, EARLY = -1.0, None, 0, 7
    exp_dir = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810"; os.makedirs(exp_dir, exist_ok=True)


- EMA keeps a smoothed version of model weights to stabilize evaluation.
- During validation, these shadow weights temporarily replace current ones.  
<br/>

- EMA는 모델 가중치의 이동 평균을 유지해, 평가 시 더 안정적인 결과를 제공합니다.
- 검증 시 EMA 가중치를 일시적으로 적용한 후 원래 모델로 복원합니다.

### Training Loop / 학습 루프

In [ ]:
    for epoch in range(1, epochs+1):
        # warmup
        if epoch <= warmup:
            wlr = base_lr * epoch / max(warmup,1)
            for g in optimizer.param_groups: g['lr'] = wlr

        model.train(); run=0.0
        pbar = tqdm(train_loader, desc=f"{tag} Epoch [{epoch}/{epochs}]", leave=False)
        for x,y in pbar:
            x,y = x.to(device), y.to(device)
            x,ya,lam,yb = mixup_data(x,y)
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                out = model(x)
                loss = mixup_crit(out, ya, yb, lam)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            run += loss.item(); pbar.set_postfix(loss=f"{loss.item():.4f}")
            ema.update(model)

        print(f"[{tag}] Avg Loss: {run/max(len(train_loader),1):.4f}")
        # EMA 평가
        backup = copy.deepcopy(model.state_dict())
        ema.apply_to(model)
        acc,f1 = evaluate(model)
        if f1>best_f1:
            best_f1, best_state, wait = f1, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
        model.load_state_dict(backup)

        # 임시 세이브(옵션)
        if tmp_save_every and epoch % tmp_save_every == 0 and best_state is not None:
            tmp_path = os.path.join(exp_dir, f"{tag}_ep{epoch}_tmp.pth")
            torch.save(best_state, tmp_path); print(f"💾 temp save: {tmp_path}")

        scheduler.step()
        memory_guard()
        if wait >= EARLY:
            print("\t>> Early stopped."); break

- Performs mixup augmentation and mixed-precision training each epoch.
- Evaluates on validation set using EMA weights.
- Supports optional temporary model saving.
- Applies early stopping if F1 doesn’t improve for 7 epochs.  
<br/>

- 각 epoch마다 mixup 증강과 AMP(혼합정밀도) 학습을 수행합니다.
- EMA 가중치 적용 상태로 검증을 수행합니다.
- 임시 모델 저장 기능이 있으며, 7 epoch 이상 F1이 개선되지 않으면 조기 종료됩니다.

### Model Saving / 모델 저장

In [ ]:

    if best_state is None: best_state = model.state_dict()
    save_path = os.path.join(exp_dir, f"{tag}.pth")
    torch.save(best_state, save_path)
    print(f"✅ 저장: {save_path}")
    print(f"📊 metrics: {{'tag': '{tag}', 'backbone': '{backbone}', 'scope': '{scope}', 'epochs': {epochs}, 'warmup': {warmup}, 'lr': {base_lr}, 'batch_size': {BATCH}, 'best_macro_f1': {best_f1}}}")
    return save_path, best_f1

- Saves the best-performing model checkpoint to the experiment directory and prints summary metrics.

- 가장 높은 성능을 기록한 모델을 지정된 경로에 저장하고, 주요 학습 결과를 콘솔에 출력합니다.

# Distillation: ResNet18 → EfficientNet-B0

## 1. Overview / 개요

This script performs knowledge distillation from a pretrained ResNet18 teacher to an EfficientNet-B0 student on a four-class emotion recognition task (angry, sad, neutral, happy).
It combines soft targets (teacher logits) and hard labels using KL divergence and label smoothing.  
<br/>

이 스크립트는 사전 학습된 ResNet18 교사 모델로부터 EfficientNet-B0 학생 모델로 감정 분류 지식을 전이하는 지식 증류를 수행합니다.
교사의 soft logits과 실제 hard label을 결합하여 학습합니다.

In [ ]:
# ==== Distillation: ResNet18(s2) -> EfficientNet-B0(student) ====
import os, time, copy, json, math
from dataclasses import dataclass
from typing import Dict, List
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from torchvision.models import resnet18, ResNet18_Weights, efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

## 2. Configuration / 설정

In [ ]:
# ---------- config ----------
@dataclass
class Cfg:
    classes: List[str] = ("angry","sad","neutral","happy")
    img_size: int = 160            # 메모리 세이프
    batch_size: int = 24           # 메모리 세이프
    grad_accum: int = 2            # 유효 배치 48
    epochs: int = 12
    warmup: int = 2
    base_lr: float = 2e-4
    weight_decay: float = 0.05
    label_smoothing: float = 0.05
    T: float = 3.0                 # KD 온도
    kd_alpha: float = 0.6          # 소프트(교사) 비중
    seed: int = 42
    num_workers: int = 0           # 메모리 세이프
    # 경로 (Star 환경)
    base_path_1: str = "/Users/star/Desktop/DSWU/2025/code/emotion_cnn/archive"
    base_path_2: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/archive"
    base_path_3: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/affectnet_filtered"
    teacher_path: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/s2.pth"
    out_dir: str = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810"
    student_out: str = "student_b0_distilled.pth"

cfg = Cfg()
os.makedirs(cfg.out_dir, exist_ok=True)

- img_size=160: smaller input for safe memory usage.
- batch_size=24, grad_accum=2: gradient accumulation effectively simulates batch size 48.
- T (Temperature)=3.0: smooths teacher logits during distillation.
- kd_alpha=0.6: weight ratio for teacher’s soft targets vs hard labels.  
<br/>

- img_size=160: 메모리 절약을 위해 작은 입력 크기 사용.
- batch_size=24, grad_accum=2: 누적 그래디언트로 유효 배치 크기 48 유지.
- T=3.0: 교사 로짓을 부드럽게 만들어 학생이 일반화된 표현을 학습하도록 함.
- kd_alpha=0.6: 교사 soft target과 hard label의 비율 설정.

## 3. Data Preparation / 데이터 전처리

In [ ]:
# ---------- seed & device ----------
def set_seed(s=42):
    import random
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)  # CPU일 때 속도/메모리 절충

# ---------- transforms (가벼움) ----------
train_tf = transforms.Compose([
    transforms.Resize(224 if cfg.img_size<=192 else 256),
    transforms.CenterCrop(cfg.img_size),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2,0.2,0.2,0.05),
    transforms.RandomGrayscale(0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
test_tf = transforms.Compose([
    transforms.Resize(224 if cfg.img_size<=192 else 256),
    transforms.CenterCrop(cfg.img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

- Uses lightweight transformations (resize, crop, color jitter, grayscale) for efficiency on small datasets.
- 가벼운 전처리(리사이즈, 센터 크롭, 색상 변형, 흑백 변환)를 사용해 CPU에서도 효율적으로 동작합니다.

In [ ]:
# ---------- dataset & relabel ----------
cls2idx = {c:i for i,c in enumerate(cfg.classes)}
def remap(ds):
    inv = {v:k for k,v in ds.class_to_idx.items()}
    new_samples, new_labels = [], []
    for p, y in ds.samples:
        name = inv[y]
        if name in cls2idx:
            ny = cls2idx[name]
            new_samples.append((p, ny)); new_labels.append(ny)
    ds.samples = new_samples; ds.targets = new_labels; ds.class_to_idx = cls2idx
    return ds, new_labels

def load_concat(root, tf, split):
    ds1 = datasets.ImageFolder(os.path.join(cfg.base_path_1, split), transform=tf)
    ds2 = datasets.ImageFolder(os.path.join(cfg.base_path_2, split), transform=tf)
    ds3 = datasets.ImageFolder(os.path.join(cfg.base_path_3, split), transform=tf)
    ds1, l1 = remap(ds1); ds2, l2 = remap(ds2); ds3, l3 = remap(ds3)
    return ConcatDataset([ds1, ds2, ds3]), (l1+l2+l3)

train_set, all_train_labels = load_concat("", train_tf, "train")
test_set, _ = load_concat("", test_tf, "test")

- Loads datasets from three sources (emotion_cnn, emotion_resnet, affectnet_filtered), remaps labels to unified four-class structure, and concatenates them.

- 세 개의 데이터 경로(emotion_cnn, emotion_resnet, affectnet_filtered)에서 데이터를 불러오고,
모든 라벨을 4개 감정 클래스로 통합한 후 ConcatDataset으로 병합합니다.

## 4. Data Loader with Weighted Sampling / 가중치 샘플링 DataLoader

In [ ]:
# sampler(가볍게)
cnt = Counter(all_train_labels)
class_w = [1.0/max(cnt.get(i,1),1) for i in range(len(cfg.classes))]
sample_w = []
for sub in train_set.datasets:
    sample_w.extend([class_w[y] for _, y in sub.samples])
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_set, batch_size=cfg.batch_size, sampler=sampler,
                          num_workers=cfg.num_workers, pin_memory=False, drop_last=True)
test_loader  = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=False)

- To handle class imbalance, each sample is weighted inversely proportional to its class frequency.
- 클래스 불균형을 완화하기 위해, 클래스 빈도의 역수를 기반으로 샘플 가중치를 설정합니다.

## 5. Teacher Model / 교사 모델

In [ ]:
# ---------- teacher ----------
assert os.path.exists(cfg.teacher_path), f"교사 가중치 없음: {cfg.teacher_path}"
teacher = resnet18(weights=ResNet18_Weights.DEFAULT)
in_f = teacher.fc.in_features
teacher.fc = nn.Linear(in_f, len(cfg.classes))
teacher.load_state_dict(torch.load(cfg.teacher_path, map_location="cpu"), strict=False)
teacher = teacher.to(device).eval()
for p in teacher.parameters(): p.requires_grad=False

- Loads the pretrained ResNet18 model from the specified teacher checkpoint (s2.pth) and freezes all parameters.
- ResNet18 기반 교사 모델(s2.pth)을 로드하고, 모든 파라미터를 고정(freeze)시켜 학습 중 변경되지 않도록 합니다.

## 6. Student Model / 학생 모델

In [ ]:
# ---------- student ----------
student = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
# head 교체
student.classifier[1] = nn.Linear(student.classifier[1].in_features, len(cfg.classes))
student = student.to(device)

- Initializes an EfficientNet-B0 model pretrained on ImageNet and replaces its classifier layer to output four emotion classes.
- ImageNet으로 사전학습된 EfficientNet-B0 모델을 불러오고,
마지막 classifier를 4개 감정 클래스 분류용으로 교체합니다.

## 7. Loss Functions / 손실 함수

In [ ]:
# ---------- losses ----------
class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.05): super().__init__(); self.eps=eps
    def forward(self, logits, target):
        n = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true = torch.zeros_like(logp).fill_(self.eps/(n-1))
            true.scatter_(1, target.unsqueeze(1), 1-self.eps)
        return torch.mean(torch.sum(-true*logp, dim=1))

ce_loss = LabelSmoothingCE(cfg.label_smoothing)

def kd_loss(student_logits, teacher_logits, T=3.0):
    # KL(student || teacher)
    ps = F.log_softmax(student_logits/T, dim=1)
    pt = F.softmax(teacher_logits/T, dim=1)
    return F.kl_div(ps, pt, reduction='batchmean') * (T*T)

optimizer = AdamW(student.parameters(), lr=cfg.base_lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)


- LabelSmoothingCE: mitigates overconfidence for hard labels.
- KD Loss: KL divergence between student and teacher logits, scaled by temperature.
- Final loss: total_loss = α * KD + (1 - α) * CE  
<br/>

- LabelSmoothingCE: hard label에 대한 과적합을 줄입니다.
- KD Loss: 교사와 학생 로짓 간의 KL divergence 계산 (온도 스케일 적용).
- 최종 손실: total_loss = α * KD + (1 - α) * CE

## 8.Evaluation Function / 평가 함수

In [ ]:
# ---------- eval ----------
@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    all_p, all_y = [], []
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        logits = net(x)
        pred = logits.argmax(1)
        all_p.extend(pred.cpu().numpy()); all_y.extend(y.cpu().numpy())
    acc = accuracy_score(all_y, all_p); mF1 = f1_score(all_y, all_p, average='macro')
    return acc, mF1

- Switches model to evaluation mode (model.eval()).
- Iterates over the validation dataloader without computing gradients.
- Collects predictions and labels to compute accuracy and macro F1-score.
- Macro F1 is used as the main metric because it balances class performance even under imbalance.  
<br/>

- 모델을 평가 모드(model.eval())로 전환합니다.
- 검증 데이터셋 전체를 순회하며 예측과 실제 라벨을 수집합니다.
- **정확도(accuracy)**와 Macro F1-score를 계산하여 성능을 평가합니다.
- Macro F1은 클래스 불균형이 있는 감정 데이터에서도 균형 잡힌 지표로 사용됩니다.

## 9.Training Loop (with Gradient Accumulation)

In [ ]:
# ---------- train (gradient accumulation) ----------
best_f1=-1.0; best_state=None
for epoch in range(1, cfg.epochs+1):
    # warmup
    if epoch<=cfg.warmup:
        for g in optimizer.param_groups:
            g['lr'] = cfg.base_lr * epoch / cfg.warmup

    student.train(); run=0.0; step=0
    pbar = tqdm(train_loader, desc=f"KD Epoch [{epoch}/{cfg.epochs}]", leave=False)
    for x,y in pbar:
        x,y = x.to(device), y.to(device)
        with torch.no_grad():
            t_logits = teacher(x)
        s_logits = student(x)

        loss_soft = kd_loss(s_logits, t_logits, T=cfg.T)
        loss_hard = ce_loss(s_logits, y)
        loss = cfg.kd_alpha*loss_soft + (1-cfg.kd_alpha)*loss_hard

        (loss/cfg.grad_accum).backward()
        step += 1
        if step % cfg.grad_accum == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        run += float(loss)
        pbar.set_postfix(loss=f"{float(loss):.4f}")

    # eval
    acc, mF1 = evaluate(student, test_loader)
    print(f"[Epoch {epoch}] loss={run/len(train_loader):.4f} | Acc={acc*100:.2f}% | Macro-F1={mF1*100:.2f}%")

    if mF1>best_f1:
        best_f1=mF1; best_state=copy.deepcopy(student.state_dict())

    scheduler.step()


- Warmup Learning Rate: Gradually increases LR during the first few epochs to stabilize early training.
- Gradient Accumulation:
  - The loss is divided by cfg.grad_accum before backpropagation.
  - After every grad_accum steps, the optimizer updates the weights.
  - This allows for effectively larger batch training with limited memory.
- Loss Composition:
  - loss_soft: KL divergence between student and teacher logits.
  - loss_hard: Label-smoothed cross-entropy with ground truth.
  - total_loss = α * loss_soft + (1 - α) * loss_hard.
- Evaluation:
  - After each epoch, evaluate the model on the test set.
  - Save the model state if Macro-F1 improves.
- Scheduler:
  - Uses cosine annealing to gradually reduce learning rate.

- Warmup Learning Rate: 초기 몇 epoch 동안 학습률을 점진적으로 올려 학습 불안정성을 줄입니다.
- Gradient Accumulation:
  - 손실값을 cfg.grad_accum으로 나눈 뒤 역전파(backward)를 수행합니다.
  - 누적된 그래디언트가 지정된 스텝 수(grad_accum)에 도달하면 가중치를 한 번 업데이트합니다.
  - GPU 메모리가 부족할 때, 큰 배치 크기를 시뮬레이션할 수 있습니다.
- 손실 구성:
  - loss_soft: 학생과 교사 로짓의 KL divergence (지식 증류 손실).
  - loss_hard: 실제 정답에 대한 라벨 스무딩 교차 엔트로피 손실.
  - 전체 손실은 α * loss_soft + (1 - α) * loss_hard로 계산됩니다.
- 평가:
  - 각 epoch마다 test set에서 모델을 평가하고 Macro-F1이 향상되면 모델 가중치를 저장합니다.
- 스케줄러:
  - CosineAnnealing을 사용해 학습률을 점진적으로 감소시킵니다.

## 10. Saving the Best Model / 최고 모델 저장

In [ ]:
# save once
save_path = os.path.join(cfg.out_dir, cfg.student_out)
torch.save(best_state if best_state is not None else student.state_dict(), save_path)
with open(os.path.join(cfg.out_dir, "student_metrics.json"), "w") as f:
    json.dump({"best_macro_f1": float(best_f1)}, f, indent=2)
print(f"\n✅ 학생 모델 저장: {save_path}\n   >> Best Macro-F1: {best_f1*100:.2f}%")

KeyboardInterrupt: 

- Saves the best-performing student model (highest Macro-F1) to the specified output directory.
- Writes a summary JSON file containing the best Macro-F1 score for record keeping.  
<br/>

- Macro-F1 기준 최고 성능을 보인 학생 모델의 가중치를 지정된 폴더에 저장합니다.
- 학습 결과를 student_metrics.json 파일로 기록해 이후 분석이나 비교에 활용할 수 있습니다.

In [8]:
# ==== KD: ResNet18(s2) 교사 -> EfficientNet-B0 학생 (MPS/CPU 호환, FP32) ====
import os, json, time, copy
from collections import Counter
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import datasets, transforms as T, models
from torchvision.models import ResNet18_Weights, EfficientNet_B0_Weights
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm

# -------------------------------
# 설정 (경로/하이퍼)
# -------------------------------
BASE1 = "/Users/star/Desktop/DSWU/2025/code/emotion_cnn/archive"
BASE2 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/archive"
BASE3 = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/affectnet_filtered"

TEACHER_PTH = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/s2.pth"
OUT_DIR     = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810"
STUDENT_OUT = "student_effb0_from_s2.pth"

TARGET_CLASSES = ("angry","sad","neutral","happy")
IMG_SIZE   = 224
BATCH_SIZE = 16           # M1 Pro 16GB 안전값
NUM_WORKERS= 0            # MPS/노트북은 0 권장
EPOCHS     = 12
WARMUP     = 2
BASE_LR    = 2e-4
WEIGHT_DECAY = 5e-2
LABEL_SMOOTH = 0.05

# KD 하이퍼
USE_KD = True
TEMP   = 3.0
ALPHA  = 0.6              # total = ALPHA*KD + (1-ALPHA)*CE

SEED = 42

# -------------------------------
# 유틸
# -------------------------------
def set_seed(seed=42):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); 
set_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(">> device:", device.type)

new_class_to_idx: Dict[str,int] = {c:i for i,c in enumerate(TARGET_CLASSES)}

def remap_labels(ds: datasets.ImageFolder):
    inv = {v:k for k,v in ds.class_to_idx.items()}
    samples, labels = [], []
    for p,y in ds.samples:
        cname = inv[y]
        if cname in new_class_to_idx:
            ny = new_class_to_idx[cname]
            samples.append((p, ny)); labels.append(ny)
    ds.samples = samples; ds.targets = labels; ds.class_to_idx = new_class_to_idx
    return ds, labels

def make_transforms(img_size=224, train=True):
    if train:
        return T.Compose([
            T.Resize(256), T.RandomResizedCrop(img_size, scale=(0.85,1.0)),
            T.RandomHorizontalFlip(0.5),
            T.ColorJitter(0.2,0.2,0.2,0.05),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])
    else:
        return T.Compose([
            T.Resize(256), T.CenterCrop(img_size),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])

def load_concat():
    tr_tf, te_tf = make_transforms(IMG_SIZE, True), make_transforms(IMG_SIZE, False)
    train_sets, test_sets, all_y = [], [], []
    for bp in [BASE1, BASE2, BASE3]:
        tr = datasets.ImageFolder(os.path.join(bp, "train"), transform=tr_tf)
        tr, ls = remap_labels(tr); train_sets.append(tr); all_y += ls
        te = datasets.ImageFolder(os.path.join(bp, "test"), transform=te_tf)
        te, _  = remap_labels(te); test_sets.append(te)
    return ConcatDataset(train_sets), ConcatDataset(test_sets), all_y

def make_loaders(full_train, full_test):
    # imbalance -> WeightedRandomSampler
    all_labels = []
    for ds in full_train.datasets: all_labels += ds.targets
    cnt = Counter(all_labels)
    class_w = [1.0/max(cnt.get(i,1),1) for i in range(len(TARGET_CLASSES))]
    sample_w = []
    for ds in full_train.datasets: sample_w += [class_w[y] for _,y in ds.samples]
    sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(full_train, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=False, drop_last=True)
    val_loader   = DataLoader(full_test, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=False)
    return train_loader, val_loader

# -------------------------------
# 모델
# -------------------------------
def build_teacher_resnet18(num_classes=4, dropout=0.3):
    m = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    in_f = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(in_f,128), nn.ReLU(True), nn.Dropout(dropout), nn.Linear(128,num_classes)
    )
    return m

def build_student_effb0(num_classes=4, dropout=0.4):
    m = models.efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_f = m.classifier[1].in_features
    if isinstance(m.classifier[0], nn.Dropout): m.classifier[0].p = dropout
    m.classifier[1] = nn.Linear(in_f, num_classes)
    return m

class LabelSmoothingCE(nn.Module):
    def __init__(self, eps=0.1): 
        super().__init__(); self.eps=eps
    def forward(self, logits, target):
        n = logits.size(1)
        logp = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true = torch.zeros_like(logp).fill_(self.eps/(n-1))
            true.scatter_(1, target.unsqueeze(1), 1-self.eps)
        return torch.mean(torch.sum(-true*logp, dim=1))

# -------------------------------
# 평가
# -------------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, gts = [], []
    for x,y in loader:
        x = x.to(device, dtype=torch.float32); y = y.to(device)
        out = model(x)
        preds += out.argmax(1).cpu().tolist()
        gts   += y.cpu().tolist()
    acc = accuracy_score(gts, preds)
    f1  = f1_score(gts, preds, average="macro")
    print(f"\tVal Acc: {acc*100:.2f}% | Macro-F1: {f1*100:.2f}%")
    return acc, f1

# -------------------------------
# 데이터 로드
# -------------------------------
full_train, full_val, all_y = load_concat()
train_loader, val_loader = make_loaders(full_train, full_val)

print(f">> train size: {len(full_train)} | val size: {len(full_val)}")

# -------------------------------
# 모델 로드 (교사)
# -------------------------------
teacher = build_teacher_resnet18(len(TARGET_CLASSES)).to(device)
state = torch.load(TEACHER_PTH, map_location="cpu")
teacher.load_state_dict(state, strict=True)
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False

# 빠른 sanity check
print(">> teacher quick eval...")
evaluate(teacher, val_loader)

# -------------------------------
# 학생 + KD 학습 세팅
# -------------------------------
student = build_student_effb0(len(TARGET_CLASSES), dropout=0.4).to(device)

criterion_ce = LabelSmoothingCE(LABEL_SMOOTH)
criterion_kd = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(student.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_f1, best_state = -1.0, None
global_step = 0

for epoch in range(1, EPOCHS+1):
    # warmup
    if epoch <= WARMUP:
        warm_lr = BASE_LR * epoch / max(WARMUP,1)
        for g in optimizer.param_groups: g["lr"] = warm_lr

    student.train()
    running = 0.0
    pbar = tqdm(train_loader, desc=f"KD Epoch [{epoch}/{EPOCHS}]", leave=False)
    for x,y in pbar:
        x = x.to(device, dtype=torch.float32)
        y = y.to(device)

        with torch.no_grad():
            t_logits = teacher(x)

        s_logits = student(x)

        loss_ce = criterion_ce(s_logits, y)
        if USE_KD:
            # KD: student vs teacher soft target
            log_p_s = F.log_softmax(s_logits / TEMP, dim=1)
            p_t     = F.softmax(t_logits / TEMP, dim=1)
            loss_kd = criterion_kd(log_p_s, p_t) * (TEMP*TEMP)
            loss = ALPHA*loss_kd + (1-ALPHA)*loss_ce
        else:
            loss = loss_ce

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

        global_step += 1

    print(f"[Epoch {epoch}] Avg Loss: {running/max(1,len(train_loader)):.4f}")

    # 평가
    _, f1 = evaluate(student, val_loader)
    if f1 > best_f1:
        best_f1 = f1
        best_state = copy.deepcopy(student.state_dict())

    scheduler.step()

# 저장
os.makedirs(OUT_DIR, exist_ok=True)
save_path = os.path.join(OUT_DIR, STUDENT_OUT)
torch.save(best_state if best_state is not None else student.state_dict(), save_path)
with open(os.path.join(OUT_DIR, "student_metrics.json"), "w") as f:
    json.dump({"best_macro_f1": float(best_f1)}, f, indent=2)

print(f"\n✅ 학생 모델 저장: {save_path}\n>> Best Macro-F1: {best_f1*100:.2f}%")

>> device: mps
>> train size: 41873 | val size: 10439
>> teacher quick eval...
	Val Acc: 84.35% | Macro-F1: 82.00%


[Epoch 1] Avg Loss: 0.4990
	Val Acc: 79.99% | Macro-F1: 77.14%


[Epoch 2] Avg Loss: 0.3729
	Val Acc: 82.59% | Macro-F1: 79.68%


[Epoch 3] Avg Loss: 0.3293
	Val Acc: 82.47% | Macro-F1: 79.82%


[Epoch 4] Avg Loss: 0.2998
	Val Acc: 83.41% | Macro-F1: 80.90%


...

[Epoch 8] Avg Loss: 0.2275
	Val Acc: 84.59% | Macro-F1: 82.20%


[Epoch 9] Avg Loss: 0.2195
	Val Acc: 84.61% | Macro-F1: 82.05%


[Epoch 10] Avg Loss: 0.2124
	Val Acc: 84.95% | Macro-F1: 82.53%


[Epoch 11] Avg Loss: 0.2062
	Val Acc: 85.30% | Macro-F1: 82.86%


[Epoch 12] Avg Loss: 0.2053
	Val Acc: 85.40% | Macro-F1: 83.02%

✅ 학생 모델 저장: /Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/student_effb0_from_s2.pth
>> Best Macro-F1: 83.02%


In [10]:
import torch, torch.nn as nn
from torchvision import models

# 경로들
PT_PATH = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/student_effb0_from_s2.pth"
ONNX_PATH = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/student_effb0_from_s2.onnx"

# 모델 정의 (학습 때와 같은 헤드)
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)  # angry,sad,neutral,happy
model.load_state_dict(torch.load(PT_PATH, map_location="cpu"))
model.eval()

dummy = torch.randn(1, 3, 224, 224)  # NCHW, float32

torch.onnx.export(
    model, dummy, ONNX_PATH,
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13, do_constant_folding=True
)

print("✅ Exported:", ONNX_PATH)

✅ Exported: /Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/student_effb0_from_s2.onnx


In [11]:
# pip install onnx onnxruntime  # (Apple Silicon이면 onnxruntime-silicon)
import onnxruntime as ort
import numpy as np, cv2

EMO = ['angry','sad','neutral','happy']
sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

def preprocess_bgr(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224,224))
    img = img.astype(np.float32) / 255.0
    mean = np.array([0.485,0.456,0.406], dtype=np.float32)
    std  = np.array([0.229,0.224,0.225], dtype=np.float32)
    img = (img - mean) / std
    img = np.transpose(img, (2,0,1))          # HWC->CHW
    return img[None, ...]                      # (1,3,224,224)

# 테스트
img = np.zeros((224,224,3), np.uint8)  # 임시
inp = preprocess_bgr(img)
logits = sess.run(["logits"], {"input": inp})[0]
pred = int(np.argmax(logits, axis=1)[0])
print("pred:", EMO[pred])

pred: angry


In [9]:
# === 실시간 감정 인식 (학생 모델: EfficientNet-B0) ===
# - Jupyter에서 바로 실행
# - q 또는 ESC로 종료
# - macOS라면 AVFoundation 백엔드를 우선 사용

import os, time, cv2, torch, torch.nn as nn
import numpy as np
from torchvision import models, transforms
from PIL import Image

# --------------------
# 설정
# --------------------
MODEL_PATH = "/Users/star/Desktop/DSWU/2025/code/emotion_resnet/exp_20250810/student_effb0_from_s2.pth"
CLASSES = ['angry', 'sad', 'neutral', 'happy']

# 디바이스 (MPS 우선)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(">> device:", device)

# --------------------
# 모델 로드 (EfficientNet-B0, 4 classes)
# --------------------
model = models.efficientnet_b0(weights=None)
in_f = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_f, len(CLASSES))
state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model.eval().to(device)

# --------------------
# 전처리 (학습 시와 동일한 ImageNet 정규화)
# --------------------
tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# --------------------
# 얼굴 검출 (Haar)
# --------------------
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# --------------------
# 예측 스무딩 (EMA)
# --------------------
ema_prob = None
ema_alpha = 0.35  # 값 클수록 최근 프레임 반영 강함

def smooth_prob(p):
    global ema_prob
    p = p.astype(np.float32)
    if ema_prob is None:
        ema_prob = p
    else:
        ema_prob = ema_alpha * p + (1 - ema_alpha) * ema_prob
    return ema_prob

# --------------------
# 카메라 오픈 (macOS 우선 백엔드 시도)
# --------------------
cap = None
for backend in [cv2.CAP_AVFOUNDATION, cv2.CAP_ANY]:
    cap = cv2.VideoCapture(0, backend)
    if cap.isOpened():
        break

if not cap or not cap.isOpened():
    raise RuntimeError("웹캠을 열 수 없습니다. 시스템 환경설정 > 보안 및 개인 정보 보호에서 카메라 권한을 확인하세요.")

print("🎥 웹캠 감정 인식 시작! (종료: q 또는 ESC)")

# --------------------
# 메인 루프
# --------------------
while True:
    ok, frame = cap.read()
    if not ok:
        print("프레임을 읽지 못했습니다.")
        break

    # 거울 모드가 편하면 주석 해제
    # frame = cv2.flip(frame, 1)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5, minSize=(80,80))

    for (x, y, w, h) in faces:
        # 컬러 영역 crop (모델은 RGB 기준)
        face_bgr = frame[y:y+h, x:x+w]
        if face_bgr.size == 0:
            continue
        face_rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)

        # 전처리
        img = Image.fromarray(face_rgb)
        inp = tf(img).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(inp)
            prob = torch.softmax(logits, dim=1).cpu().numpy()[0]

        prob = smooth_prob(prob)
        pred_idx = int(np.argmax(prob))
        label = CLASSES[pred_idx]
        conf = float(prob[pred_idx])

        # 시각화
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 200, 0), 2)
        text = f"{label}: {conf*100:5.1f}%"
        cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.80, (0, 200, 0), 2, cv2.LINE_AA)

        # 확률 바(하단)
        bar_x, bar_y = x, y+h+22
        bar_w, bar_h = w, 18
        cv2.rectangle(frame, (bar_x, bar_y), (bar_x+bar_w, bar_y+bar_h), (30,30,30), -1)
        # 각각 클래스 확률을 작은 막대로
        step = max(1, int(bar_w/len(CLASSES)))
        for i, p in enumerate(prob):
            pw = int(step * p)
            cv2.rectangle(frame,
                          (bar_x + i*step, bar_y),
                          (bar_x + i*step + pw, bar_y + bar_h),
                          (0, 160, 255) if i==pred_idx else (80,80,80),
                          -1)
            cv2.putText(frame, CLASSES[i][0].upper(), (bar_x + i*step + 2, bar_y + bar_h - 4),
                        cv2.FONT_HERSHEY_PLAIN, 0.9, (255,255,255), 1, cv2.LINE_AA)

    cv2.imshow("Emotion Detection (EffB0 student)", frame)
    k = cv2.waitKey(1) & 0xFF
    if k in [27, ord('q')]:  # ESC or q
        break

cap.release()
cv2.destroyAllWindows()

>> device: mps


2025-08-11 22:56:16.185 python[2670:44771] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


🎥 웹캠 감정 인식 시작! (종료: q 또는 ESC)
